In [2]:
"/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"

'/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv'

In [4]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

# Q1 — Data Quality Audit

In [6]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

print("Shape:", df.shape)

print("\nDatatypes:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nUnique Vehicle IDs:")
print(df["vehicle_id"].nunique())

for col in ["brand", "vehicle_type", "fast_charging", "city", "service_history"]:
    print("\n", col)
    print(df[col].value_counts(dropna=False))


Shape: (1000, 15)

Datatypes:
vehicle_id               object
listing_date             object
manufacture_year          int64
brand                    object
vehicle_type             object
battery_capacity_kwh    float64
battery_health_pct      float64
range_km                float64
km_driven               float64
charging_time_hr        float64
fast_charging            object
owner_count               int64
city                     object
service_history          object
resale_price              int64
dtype: object

Missing Values:
vehicle_id               0
listing_date             0
manufacture_year         0
brand                    0
vehicle_type             0
battery_capacity_kwh    25
battery_health_pct      30
range_km                20
km_driven                0
charging_time_hr        24
fast_charging            0
owner_count              0
city                     0
service_history         30
resale_price             0
dtype: int64

Duplicate Records:
0

Unique Vehicle IDs

# Q2 — Duplicate Vehicle Investigation

In [7]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

duplicates = df[df["vehicle_id"].duplicated(keep=False)]

print("Duplicated vehicle_id values:")
print(df["vehicle_id"].duplicated().sum())

print("\nDuplicated Vehicle IDs:")
print(duplicates["vehicle_id"].value_counts())

print("\nAffected Records:")
print(len(duplicates))

print("\nDuplicate Records:")
print(duplicates)


Duplicated vehicle_id values:
6

Duplicated Vehicle IDs:
vehicle_id
EV-20515    2
EV-20653    2
EV-20011    2
EV-20544    2
EV-20193    2
EV-20279    2
Name: count, dtype: int64

Affected Records:
12

Duplicate Records:
    vehicle_id listing_date  manufacture_year       brand vehicle_type  \
72    EV-20515   2025-03-14              2016  GreenDrive    Hatchback   
81    EV-20653   2025-03-23              2019     Atheron    Crossover   
119   EV-20011   2025-04-30              2016      Nexora        Sedan   
133   EV-20544   2025-05-14              2024    E-Motion        Sedan   
280   EV-20544   2025-10-08              2024    E-Motion        Sedan   
621   EV-20515   2026-09-14              2016  GreenDrive    Hatchback   
655   EV-20193   2026-10-18              2023     Atheron    Crossover   
797   EV-20193   2027-03-09              2023     Atheron    Crossover   
816   EV-20279   2027-03-28              2019    E-Motion          SUV   
852   EV-20011   2027-05-03             

# Q3 — Date Conversion & Validation

In [8]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df["listing_date"] = pd.to_datetime(
    df["listing_date"],
    errors="coerce"
)

print("Earliest Date:", df["listing_date"].min())
print("Latest Date:", df["listing_date"].max())

print("Invalid/Missing Dates:")
print(df["listing_date"].isnull().sum())

print("\nDatatype:")
print(df["listing_date"].dtype)


Earliest Date: 2025-01-01 00:00:00
Latest Date: 2027-09-27 00:00:00
Invalid/Missing Dates:
0

Datatype:
datetime64[ns]


# Q4 — Vehicle Age Feature

In [9]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df["listing_date"] = pd.to_datetime(
    df["listing_date"],
    errors="coerce"
)

df["listing_year"] = df["listing_date"].dt.year

df["vehicle_age"] = (
    df["listing_year"] -
    df["manufacture_year"]
)

print(df["vehicle_age"].describe())

print("\nAge < 0:", (df["vehicle_age"] < 0).sum())
print("Age = 0:", (df["vehicle_age"] == 0).sum())

print("\nUnusually High Age:")
print(
    df[df["vehicle_age"] > 20][
        ["vehicle_id", "vehicle_age"]
    ]
)


count    1000.000000
mean        5.426000
std         2.992571
min         0.000000
25%         3.000000
50%         6.000000
75%         8.000000
max        11.000000
Name: vehicle_age, dtype: float64

Age < 0: 0
Age = 0: 40

Unusually High Age:
Empty DataFrame
Columns: [vehicle_id, vehicle_age]
Index: []


# Q5 — Battery Data Imputation

In [10]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

columns = [
    "battery_capacity_kwh",
    "battery_health_pct",
    "range_km",
    "charging_time_hr"
]

print("Missing Before:")
print(df[columns].isnull().sum())

for col in columns:
    df[col] = df[col].fillna(df[col].median())

print("\nMissing After:")
print(df[columns].isnull().sum())

Missing Before:
battery_capacity_kwh    25
battery_health_pct      30
range_km                20
charging_time_hr        24
dtype: int64

Missing After:
battery_capacity_kwh    0
battery_health_pct      0
range_km                0
charging_time_hr        0
dtype: int64


# Q6 — Battery Health Outlier Investigation

In [11]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

print(df["battery_health_pct"].describe())

print("\nBelow 0:")
print((df["battery_health_pct"] < 0).sum())

print("Above 100:")
print((df["battery_health_pct"] > 100).sum())

Q1 = df["battery_health_pct"].quantile(0.25)
Q3 = df["battery_health_pct"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("\nExtreme Values:")
print(
    df[
        (df["battery_health_pct"] < lower) |
        (df["battery_health_pct"] > upper)
    ][
        ["vehicle_id", "battery_health_pct"]
    ]
)

count    970.000000
mean      91.221546
std        4.835603
min       75.800000
25%       87.900000
50%       91.200000
75%       94.600000
max      100.000000
Name: battery_health_pct, dtype: float64

Below 0:
0
Above 100:
0

Extreme Values:
    vehicle_id  battery_health_pct
135   EV-20865                77.0
234   EV-20780                75.8
343   EV-20853                77.0
631   EV-20700                77.3


# Q7 — Range vs Battery Capacity

In [12]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df["range_per_kwh"] = (
    df["range_km"] /
    df["battery_capacity_kwh"]
)

print(df["range_per_kwh"].describe())

Q1 = df["range_per_kwh"].quantile(0.25)
Q3 = df["range_per_kwh"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("\nUnusual Efficiency:")
print(
    df[
        (df["range_per_kwh"] < lower) |
        (df["range_per_kwh"] > upper)
    ][
        ["vehicle_id", "range_per_kwh"]
    ]
)

count    955.000000
mean       6.765909
std        2.422140
min        2.340094
25%        5.091170
50%        6.385069
75%        8.070697
max       19.200000
Name: range_per_kwh, dtype: float64

Unusual Efficiency:
    vehicle_id  range_per_kwh
6     EV-20961      13.409742
21    EV-20934      15.501859
31    EV-20891      12.891986
133   EV-20544      13.447368
139   EV-20278      12.770270
183   EV-20789      13.223529
194   EV-20480      15.384615
223   EV-20040      16.078431
280   EV-20544      13.447368
345   EV-20186      12.762646
349   EV-20490      12.939560
376   EV-20295      16.600000
395   EV-20690      13.074627
496   EV-20650      18.517110
545   EV-20746      14.720000
570   EV-20388      13.500000
596   EV-20570      13.149351
618   EV-20479      15.132979
633   EV-20979      12.967914
635   EV-20633      15.064935
714   EV-20785      15.559322
760   EV-20086      19.200000
761   EV-20231      13.952880
772   EV-20429      12.836257
845   EV-20791      13.760000
906

# Q8 — Driving Intensity Feature

In [1]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df["listing_date"] = pd.to_datetime(
    df["listing_date"],
    errors="coerce"
)

df["listing_year"] = df["listing_date"].dt.year

df["vehicle_age"] = (
    df["listing_year"] -
    df["manufacture_year"]
)

df["km_per_year"] = df["km_driven"].div(
    df["vehicle_age"].replace(0, pd.NA)
)

print(df["km_per_year"].describe())

Q1 = df["km_per_year"].quantile(0.25)
Q3 = df["km_per_year"].quantile(0.75)

IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

print("\nHigh Annual Driving:")
print(
    df[df["km_per_year"] > upper][
        ["vehicle_id", "km_per_year"]
    ]
)

count       960.0
unique      955.0
top       60000.0
freq          3.0
Name: km_per_year, dtype: float64

High Annual Driving:
    vehicle_id km_per_year
15    EV-20445     51040.5
23    EV-20663     47597.0
28    EV-20997     52274.5
35    EV-20163     69388.0
44    EV-20493     45848.0
..         ...         ...
844   EV-20722     39228.0
864   EV-20975     44049.0
886   EV-20718     89327.5
914   EV-20496     50190.0
983   EV-20062     48874.0

[84 rows x 2 columns]


# Q9 — Charging Efficiency

In [2]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df["charging_efficiency"] = (
    df["range_km"] /
    df["charging_time_hr"]
)

print(df["charging_efficiency"].describe())

print("\nMissing Values:")
print(df["charging_efficiency"].isnull().sum())

Q1 = df["charging_efficiency"].quantile(0.25)
Q3 = df["charging_efficiency"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("\nExtreme Values:")
print(
    df[
        (df["charging_efficiency"] < lower) |
        (df["charging_efficiency"] > upper)
    ][
        ["vehicle_id", "charging_efficiency"]
    ]
)

count    956.000000
mean      66.976136
std       30.117138
min       20.888889
25%       47.017091
50%       59.742102
75%       79.061684
max      230.000000
Name: charging_efficiency, dtype: float64

Missing Values:
44

Extreme Values:
    vehicle_id  charging_efficiency
27    EV-20085           146.206897
133   EV-20544           164.838710
147   EV-20420           175.600000
166   EV-20786           207.407407
171   EV-20655           169.500000
185   EV-20355           131.428571
243   EV-20339           135.454545
267   EV-20551           159.130435
280   EV-20544           164.838710
327   EV-20871           166.923077
334   EV-20727           149.166667
365   EV-20516           157.727273
374   EV-20484           145.357143
397   EV-20307           128.055556
445   EV-20682           144.871795
466   EV-20151           170.606061
496   EV-20650           180.370370
571   EV-20776           159.062500
591   EV-20894           151.176471
597   EV-20173           229.500000
618  

# Q10 — Ownership Analysis

In [3]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

print("Owner Count Frequency:")
print(
    df["owner_count"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nInvalid Owner Count:")
print(
    df[
        (df["owner_count"] <= 0) |
        (df["owner_count"].isnull())
    ][
        ["vehicle_id", "owner_count"]
    ]
)

print("\nDatatype:")
print(df["owner_count"].dtype)

Owner Count Frequency:
owner_count
1    532
2    302
3    126
4     40
Name: count, dtype: int64

Invalid Owner Count:
Empty DataFrame
Columns: [vehicle_id, owner_count]
Index: []

Datatype:
int64


# Q11 — Categorical Data Audit

In [4]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

columns = [
    "brand",
    "vehicle_type",
    "fast_charging",
    "city",
    "service_history"
]

for col in columns:
    print("\n--------------------")
    print(col)
    print("--------------------")

    print("Unique:", df[col].nunique(dropna=False))
    print("Missing:", df[col].isnull().sum())
    print(df[col].value_counts(dropna=False))


--------------------
brand
--------------------
Unique: 5
Missing: 0
brand
Nexora        213
Atheron       207
GreenDrive    196
E-Motion      192
Voltix        192
Name: count, dtype: int64

--------------------
vehicle_type
--------------------
Unique: 4
Missing: 0
vehicle_type
SUV          265
Hatchback    251
Crossover    244
Sedan        240
Name: count, dtype: int64

--------------------
fast_charging
--------------------
Unique: 2
Missing: 0
fast_charging
Yes    692
No     308
Name: count, dtype: int64

--------------------
city
--------------------
Unique: 6
Missing: 0
city
Pune         193
Delhi        174
Chennai      168
Bengaluru    160
Mumbai       155
Hyderabad    150
Name: count, dtype: int64

--------------------
service_history
--------------------
Unique: 4
Missing: 30
service_history
Complete    590
Partial     297
Missing      83
NaN          30
Name: count, dtype: int64


# Q12 — Service History Cleaning

In [5]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

print("Before:")
print(df["service_history"].value_counts(dropna=False))

df["service_history"] = (
    df["service_history"]
    .astype("string")
    .str.strip()
    .str.title()
    .fillna("Missing")
)

print("\nAfter:")
print(df["service_history"].value_counts(dropna=False))

Before:
service_history
Complete    590
Partial     297
Missing      83
NaN          30
Name: count, dtype: int64

After:
service_history
Complete    590
Partial     297
Missing     113
Name: count, dtype: Int64


# Q13 — Fast Charging Transformation

In [6]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

print("Before:")
print(df["fast_charging"].value_counts(dropna=False))

df["fast_charging"] = (
    df["fast_charging"]
    .astype("string")
    .str.strip()
    .str.title()
    .map({
        "Yes": 1,
        "No": 0
    })
)

print("\nAfter:")
print(df["fast_charging"].value_counts(dropna=False))

Before:
fast_charging
Yes    692
No     308
Name: count, dtype: int64

After:
fast_charging
1    692
0    308
Name: count, dtype: int64


# Q14 — Target Analysis

In [7]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

print("Mean:", df["resale_price"].mean())
print("Median:", df["resale_price"].median())
print("Minimum:", df["resale_price"].min())
print("Maximum:", df["resale_price"].max())
print("Standard Deviation:", df["resale_price"].std())

Q1 = df["resale_price"].quantile(0.25)
Q3 = df["resale_price"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("\nPotential Extreme Values:")
print(
    df[
        (df["resale_price"] < lower) |
        (df["resale_price"] > upper)
    ][
        ["vehicle_id", "resale_price"]
    ]
)

Mean: 1586535.101
Median: 1589594.0
Minimum: 915109
Maximum: 2072547
Standard Deviation: 166204.3759912459

Potential Extreme Values:
    vehicle_id  resale_price
1     EV-20530       2040760
135   EV-20865       1117512
213   EV-20505       1047190
218   EV-20625       1011244
234   EV-20780        915109
597   EV-20173       2029110
665   EV-20781       2042543
786   EV-20313       1019997
873   EV-20743       2072547
963   EV-20513       1140605


# Q15 — Price-per-Kilometer Feature

In [8]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df["price_per_km"] = (
    df["resale_price"] /
    df["km_driven"].replace(0, pd.NA)
)

print(df["price_per_km"].describe())

Q1 = df["price_per_km"].quantile(0.25)
Q3 = df["price_per_km"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("\nExtreme Values:")
print(
    df[
        (df["price_per_km"] < lower) |
        (df["price_per_km"] > upper)
    ][
        ["vehicle_id", "resale_price",
         "km_driven", "price_per_km"]
    ]
)

count    1000.000000
mean       47.898997
std        38.160068
min         5.618022
25%        22.713155
50%        37.871836
75%        61.903746
max       296.870220
Name: price_per_km, dtype: float64

Extreme Values:
    vehicle_id  resale_price  km_driven  price_per_km
2     EV-20654       1830249    12003.0    152.482629
3     EV-20935       1726422    13623.0    126.728474
6     EV-20961       1605157    11615.0    138.196901
7     EV-20584       1589566     6999.0    227.113302
14    EV-20541       1889781    11531.0    163.887000
39    EV-20903       1939616    14228.0    136.323868
84    EV-20238       1886520    13442.0    140.345187
88    EV-20955       1726590     8232.0    209.741254
91    EV-20607       1856159    13806.0    134.445821
98    EV-20254       1693139    13154.0    128.716664
99    EV-20488       1605711     6812.0    235.717998
104   EV-20777       1578645     9901.0    159.442986
106   EV-20026       1821676    14546.0    125.235529
156   EV-20051       173

# Q16 — Battery Health × Usage Feature

In [9]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df["battery_usage_index"] = (
    df["km_driven"] /
    (df["battery_health_pct"] / 100)
)

print(
    df[
        ["vehicle_id",
         "km_driven",
         "battery_health_pct",
         "battery_usage_index"]
    ].head()
)

print("\nStatistics:")
print(df["battery_usage_index"].describe())

  vehicle_id    km_driven  battery_health_pct  battery_usage_index
0   EV-20170   28454.0000                89.6         31756.696429
1   EV-20530  183935.8448                99.5        184860.145528
2   EV-20654   12003.0000                91.2         13161.184211
3   EV-20935   13623.0000                89.3         15255.319149
4   EV-20827   28548.0000                92.0         31030.434783

Statistics:
count       970.000000
mean      57770.234573
std       40122.665775
min        6502.136752
25%       29080.884242
50%       46091.906154
75%       73454.157435
max      260000.948401
Name: battery_usage_index, dtype: float64


# Q17 — Complete Missing-Value Pipeline

In [10]:
import pandas as pd

df_original = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df_clean = df_original.copy()

numeric_columns = [
    "battery_capacity_kwh",
    "battery_health_pct",
    "range_km",
    "charging_time_hr"
]

for col in numeric_columns:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

df_clean["service_history"] = (
    df_clean["service_history"]
    .astype("string")
    .str.strip()
    .str.title()
    .fillna("Missing")
)

print("Missing Values After Cleaning:")
print(
    df_clean[
        numeric_columns + ["service_history"]
    ].isnull().sum()
)

print("\nOriginal Shape:", df_original.shape)
print("Cleaned Shape:", df_clean.shape)

Missing Values After Cleaning:
battery_capacity_kwh    0
battery_health_pct      0
range_km                0
charging_time_hr        0
service_history         0
dtype: int64

Original Shape: (1000, 15)
Cleaned Shape: (1000, 15)


# Q18 — Regression Feature Preparation

In [11]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

# Convert date
df["listing_date"] = pd.to_datetime(
    df["listing_date"],
    errors="coerce"
)

# Date-derived features
df["listing_year"] = df["listing_date"].dt.year
df["listing_month"] = df["listing_date"].dt.month
df["listing_quarter"] = df["listing_date"].dt.quarter

df["vehicle_age"] = (
    df["listing_year"] -
    df["manufacture_year"]
)

# Target
y = df["resale_price"]

# Features
X = df.drop(
    columns=[
        "vehicle_id",
        "resale_price",
        "listing_date"
    ]
)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nNumerical Features:")
print(
    X.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()
)

print("\nCategorical Features:")
print(
    X.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()
)

X Shape: (1000, 16)
y Shape: (1000,)

Numerical Features:
['manufacture_year', 'battery_capacity_kwh', 'battery_health_pct', 'range_km', 'km_driven', 'charging_time_hr', 'owner_count', 'vehicle_age']

Categorical Features:
['brand', 'vehicle_type', 'fast_charging', 'city', 'service_history']


# Q19 — Leakage Investigation

In [12]:
import pandas as pd

df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

df["price_per_km"] = (
    df["resale_price"] /
    df["km_driven"].replace(0, pd.NA)
)

features = [
    "vehicle_age",
    "range_per_kwh",
    "km_per_year",
    "charging_efficiency",
    "battery_usage_index",
    "price_per_km"
]

print("Leakage Check:")

for feature in features:
    if feature == "price_per_km":
        print(feature, "-> TARGET LEAKAGE")
    else:
        print(feature, "-> Safe")

# Remove leakage
X = df.drop(
    columns=[
        "vehicle_id",
        "resale_price",
        "price_per_km"
    ],
    errors="ignore"
)

y = df["resale_price"]

print("\nFinal X Columns:")
print(X.columns.tolist())

Leakage Check:
vehicle_age -> Safe
range_per_kwh -> Safe
km_per_year -> Safe
charging_efficiency -> Safe
battery_usage_index -> Safe
price_per_km -> TARGET LEAKAGE

Final X Columns:
['listing_date', 'manufacture_year', 'brand', 'vehicle_type', 'battery_capacity_kwh', 'battery_health_pct', 'range_km', 'km_driven', 'charging_time_hr', 'fast_charging', 'owner_count', 'city', 'service_history']


# Q20 — Ultimate Hard Problem

In [14]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(
    "/Users/user/Downloads/EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv"
)

# Preserve original
df_original = df.copy()

# Remove duplicate vehicle IDs
df["missing_count"] = df.isnull().sum(axis=1)

df = df.sort_values(
    ["vehicle_id", "missing_count"]
)

df = df.drop_duplicates(
    subset="vehicle_id",
    keep="first"
)

df = df.drop(columns="missing_count")

# Date cleaning
df["listing_date"] = pd.to_datetime(
    df["listing_date"],
    errors="coerce"
)

df["listing_year"] = df["listing_date"].dt.year
df["listing_month"] = df["listing_date"].dt.month
df["listing_quarter"] = df["listing_date"].dt.quarter

# Vehicle age
df["vehicle_age"] = (
    df["listing_year"] -
    df["manufacture_year"]
)

df.loc[df["vehicle_age"] < 0, "vehicle_age"] = np.nan

df["vehicle_age"] = df["vehicle_age"].fillna(
    df["vehicle_age"].median()
)

# Numerical cleaning
numeric_columns = [
    "battery_capacity_kwh",
    "battery_health_pct",
    "range_km",
    "km_driven",
    "charging_time_hr",
    "owner_count"
]

for col in numeric_columns:
    df[col] = df[col].fillna(
        df[col].median()
    )

# Battery health
df.loc[
    ~df["battery_health_pct"].between(0, 100),
    "battery_health_pct"
] = df["battery_health_pct"].median()

# Categorical cleaning
categorical_columns = [
    "brand",
    "vehicle_type",
    "fast_charging",
    "city",
    "service_history"
]

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.title()
    )

# Service history
df["service_history"] = (
    df["service_history"]
    .fillna("Missing")
)

# Fast charging
df["fast_charging"] = df[
    "fast_charging"
].map({
    "Yes": 1,
    "No": 0
})

# Feature engineering
df["range_per_kwh"] = (
    df["range_km"] /
    df["battery_capacity_kwh"]
)

df["km_per_year"] = df["km_driven"].div(
    df["vehicle_age"].replace(0, pd.NA)
)

df["charging_efficiency"] = (
    df["range_km"] /
    df["charging_time_hr"]
)

df["battery_usage_index"] = (
    df["km_driven"] /
    (df["battery_health_pct"] / 100)
)

# Replace infinite values
df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Fill engineered missing values
for col in [
    "range_per_kwh",
    "km_per_year",
    "charging_efficiency",
    "battery_usage_index"
]:
    df[col] = df[col].fillna(
        df[col].median()
    )

# Create X and y
y = df["resale_price"]

X = df.drop(
    columns=[
        "vehicle_id",
        "resale_price",
        "listing_date"
    ]
)

# Final validation
print("Final X Shape:", X.shape)
print("Final y Shape:", y.shape)

print("\nMissing Values:")
print(X.isnull().sum())

print("\nDuplicate Vehicle IDs:")
print(df["vehicle_id"].duplicated().sum())

print("\nInvalid Battery Health:")
print(
    (
        (df["battery_health_pct"] < 0) |
        (df["battery_health_pct"] > 100)
    ).sum()
)

print("\nImpossible Vehicle Age:")
print((df["vehicle_age"] < 0).sum())

print("\nX Columns:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Final X Shape: (994, 20)
Final y Shape: (994,)

Missing Values:
manufacture_year        0
brand                   0
vehicle_type            0
battery_capacity_kwh    0
battery_health_pct      0
range_km                0
km_driven               0
charging_time_hr        0
fast_charging           0
owner_count             0
city                    0
service_history         0
listing_year            0
listing_month           0
listing_quarter         0
vehicle_age             0
range_per_kwh           0
km_per_year             0
charging_efficiency     0
battery_usage_index     0
dtype: int64

Duplicate Vehicle IDs:
0

Invalid Battery Health:
0

Impossible Vehicle Age:
0

X Columns:
['manufacture_year', 'brand', 'vehicle_type', 'battery_capacity_kwh', 'battery_health_pct', 'range_km', 'km_driven', 'charging_time_hr', 'fast_charging', 'owner_count', 'city', 'service_history', 'listing_year', 'listing_month', 'listing_quarter', 'vehicle_age', 'range_per_kwh', 'km_per_year', 'charging_effici

/var/folders/13/1yf32tnn7sxcttdzt3_x3jg80000gn/T/ipykernel_6412/2643782503.py:133: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(
